In [3]:
!pip install torch_geometric

In [4]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_
from torch.amp import autocast, GradScaler
from pathlib import Path
from collections import defaultdict
import numpy as np
import random
import time

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Paths ─────────────────────────────────────────────────────────────────────
GRAPH_PATH = Path("D:/RecSys/outputs/embedded_graph.pt")
UB_PATH    = Path("D:/RecSys/outputs/edges_user_business.txt")
BB_PATH    = Path("D:/RecSys/outputs/edges_business_business_similar.txt")
CKPT_DIR   = Path("D:/RecSys/outputs/chkpt2")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Resume config ─────────────────────────────────────────────────────────────
AUTO_RESUME = True                 # True: tự tìm checkpoint epoch mới nhất trong CKPT_DIR
RESUME_FROM = None                 # Ví dụ: CKPT_DIR / "epoch_030.pt"; None = dùng AUTO_RESUME

# ── Hyperparameters ───────────────────────────────────────────────────────────
IN_DIM      = 384
EMBED_DIM   = 128
NUM_LAYERS  = 3

EPOCHS          = 80
BATCH_SIZE      = 4096
MAX_STEPS_EPOCH = 128          # tăng từ 64 → 128 để tận dụng VRAM
LR          = 1e-3
LR_DECAY    = 0.8
LR_STEP     = 15
TEMPERATURE = 0.07
LAMBDA_SIM  = 0.5
LAMBDA_USER = 1.0
DROPOUT     = 0.1
SAVE_EVERY  = 5
AMP         = torch.cuda.is_available()

# ── Curriculum schedule ───────────────────────────────────────────────────────
CURRICULUM_MIXED_START = 11
CURRICULUM_HARD_START  = 31

# ── Dynamic hard negative rebuild ─────────────────────────────────────────────
REBUILD_NEG_EVERY  = 10    # rebuild hard_neg_map từ biz_h hiện tại mỗi N epoch
REBUILD_NEG_TOP_K  = 15    # số neighbor lấy mỗi business khi rebuild
LR_HARD_RESTART    = 3e-4  # LR warm restart khi bước vào hard phase

print("Config loaded.")
print(f"  IN_DIM={IN_DIM}, EMBED_DIM={EMBED_DIM}, NUM_LAYERS={NUM_LAYERS}")
print(f"  EPOCHS={EPOCHS}, BATCH={BATCH_SIZE}, MAX_STEPS={MAX_STEPS_EPOCH}")
print(f"  LR={LR}, decay={LR_DECAY}/step{LR_STEP}, τ={TEMPERATURE}")
print(f"  λ_sim={LAMBDA_SIM}, λ_user={LAMBDA_USER}")
print(f"  Curriculum: random<{CURRICULUM_MIXED_START} | mixed<{CURRICULUM_HARD_START} | hard≥{CURRICULUM_HARD_START}")
print(f"  Dynamic neg rebuild every {REBUILD_NEG_EVERY} epochs (top_k={REBUILD_NEG_TOP_K})")
print(f"  Resume: AUTO_RESUME={AUTO_RESUME}, RESUME_FROM={RESUME_FROM}")
print(f"  AMP={'ON' if AMP else 'OFF'}")

Device: cuda
Config loaded.
  IN_DIM=384, EMBED_DIM=128, NUM_LAYERS=3
  EPOCHS=80, BATCH=4096, MAX_STEPS=128
  LR=0.001, decay=0.8/step15, τ=0.07
  λ_sim=0.5, λ_user=1.0
  Curriculum: random<11 | mixed<31 | hard≥31
  Dynamic neg rebuild every 10 epochs (top_k=15)
  Resume: AUTO_RESUME=True, RESUME_FROM=None
  AMP=ON


In [5]:
# ── 2. Load graph + build lookup maps ────────────────────────────────────────
print("Loading graph ...")
t0 = time.time()

ckpt    = torch.load(GRAPH_PATH, weights_only=False)
data    = ckpt["data"].to(DEVICE)
user2idx = ckpt["user2idx"]   # {user_id_str: int}
biz2idx  = ckpt["biz2idx"]    # {biz_id_str:  int}
idx2biz  = {v: k for k, v in biz2idx.items()}
idx2user = {v: k for k, v in user2idx.items()}

NUM_USERS = data["user"].num_nodes
NUM_BIZ   = data["business"].num_nodes
print(f"  Users     : {NUM_USERS:,}")
print(f"  Businesses: {NUM_BIZ:,}")
print(f"  Edge types: {data.edge_types}")

# ── User positive set: {user_idx: set(biz_idx)} ──────────────────────────────
print("\nBuilding user positive set ...")
user_pos: dict[int, set] = defaultdict(set)
with UB_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2 and parts[0] in user2idx and parts[1] in biz2idx:
            user_pos[user2idx[parts[0]]].add(biz2idx[parts[1]])
print(f"  Users with positives: {len(user_pos):,}")

# ── Hard negative map: {biz_idx: [similar_biz_idx]} ──────────────────────────
print("\nBuilding hard negative map ...")
hard_neg_map: dict[int, list] = defaultdict(list)
with BB_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2 and parts[0] in biz2idx and parts[1] in biz2idx:
            hard_neg_map[biz2idx[parts[0]]].append(biz2idx[parts[1]])

# ── SimDataset pairs: list of (anchor_idx, pos_idx) ──────────────────────────
print("\nLoading BB similar pairs ...")
sim_pairs: list[tuple[int,int]] = []
with BB_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2 and parts[0] in biz2idx and parts[1] in biz2idx:
            sim_pairs.append((biz2idx[parts[0]], biz2idx[parts[1]]))

# ── UB pairs: list of (user_idx, biz_idx) ────────────────────────────────────
print("\nLoading UB pairs ...")
ub_pairs: list[tuple[int,int]] = []
for u_idx, pos_set in user_pos.items():
    for b_idx in pos_set:
        ub_pairs.append((u_idx, b_idx))

elapsed = time.time() - t0
print(f"\nSummary:")
print(f"  sim_pairs : {len(sim_pairs):>10,}")
print(f"  ub_pairs  : {len(ub_pairs):>10,}")
print(f"  hard_neg  : {len(hard_neg_map):>10,} businesses have neighbors")
print(f"  Loaded in : {elapsed:.1f}s")


Loading graph ...


FileNotFoundError: [Errno 2] No such file or directory: 'D:/RecSys/outputs/embedded_graph.pt'

In [ ]:
# ── 3. Model: HeteroLightGCN ─────────────────────────────────────────────────

def make_norm_adj(edge_index: torch.Tensor, n_src: int, n_dst: int) -> torch.Tensor:
    """
    Build symmetric-normalized sparse adjacency (n_dst × n_src).
    D_src^{-1/2} * A * D_dst^{-1/2}
    """
    row = edge_index[0].cpu()
    col = edge_index[1].cpu()

    deg_dst = torch.zeros(n_dst).scatter_add_(0, col, torch.ones(col.size(0)))
    deg_src = torch.zeros(n_src).scatter_add_(0, row, torch.ones(row.size(0)))

    d_dst_inv_sqrt = deg_dst.pow(-0.5).clamp(max=1e5)
    d_src_inv_sqrt = deg_src.pow(-0.5).clamp(max=1e5)

    val     = d_dst_inv_sqrt[col] * d_src_inv_sqrt[row]
    indices = torch.stack([col, row], dim=0)
    adj = torch.sparse_coo_tensor(indices, val, (n_dst, n_src)).coalesce()
    return adj


class HeteroLightGCN(nn.Module):
    """
    LightGCN trên đồ thị heterogeneous.
    - Linear projection: SentenceTransformer 384 → EMBED_DIM
    - num_layers lớp propagation không tham số
    - Output = mean pooling tất cả các lớp (kể cả lớp 0)
    - Projection head (SimHead) cho InfoNCE
    - Sparse matmul luôn chạy fp32 (CUDA sparse không hỗ trợ fp16)
    """
    def __init__(self, in_dim: int, embed_dim: int, num_layers: int, dropout: float = 0.1):
        super().__init__()
        self.num_layers = num_layers
        self.dropout    = nn.Dropout(dropout)

        self.user_proj = nn.Linear(in_dim, embed_dim, bias=False)
        self.biz_proj  = nn.Linear(in_dim, embed_dim, bias=False)

        self.sim_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )

        nn.init.xavier_uniform_(self.user_proj.weight)
        nn.init.xavier_uniform_(self.biz_proj.weight)

    def forward(
        self,
        user_feat: torch.Tensor,
        biz_feat:  torch.Tensor,
        adj_ub:    torch.Tensor,   # (N_biz,  N_user) sparse float32
        adj_bu:    torch.Tensor,   # (N_user, N_biz)  sparse float32
        adj_uu:    torch.Tensor,   # (N_user, N_user) sparse float32
        adj_bb:    torch.Tensor,   # (N_biz,  N_biz)  sparse float32
    ):
        # Layer-0: projected features (may be fp16 under autocast)
        u0 = self.dropout(self.user_proj(user_feat))
        b0 = self.dropout(self.biz_proj(biz_feat))

        u_layers = [u0]
        b_layers = [b0]

        u_cur, b_cur = u0, b0
        for _ in range(self.num_layers):
            # Sparse matmul requires fp32 on CUDA → cast dense tensors, then cast back
            u_f = u_cur.float()
            b_f = b_cur.float()
            u_new = (adj_bu @ b_f + adj_uu @ u_f).to(u_cur.dtype)
            b_new = (adj_ub @ u_f + adj_bb @ b_f).to(b_cur.dtype)

            u_cur, b_cur = u_new, b_new
            u_layers.append(u_cur)
            b_layers.append(b_cur)

        user_emb = torch.stack(u_layers, dim=0).mean(dim=0)
        biz_emb  = torch.stack(b_layers, dim=0).mean(dim=0)

        user_h = F.normalize(user_emb, dim=-1)
        biz_h  = F.normalize(biz_emb,  dim=-1)

        biz_proj  = F.normalize(self.sim_head(biz_emb),  dim=-1)
        user_proj = F.normalize(self.sim_head(user_emb), dim=-1)

        return user_h, biz_h, user_proj, biz_proj


# ── Build sparse adjacency matrices ──────────────────────────────────────────
print("Building sparse adjacency matrices ...")
_t = time.time()

ub_ei  = data[("user",     "rates",       "business")].edge_index
bu_ei  = data[("business", "rated_by",    "user")    ].edge_index
uu_ei  = data[("user",     "friends",     "user")    ].edge_index
bb_ei1 = data[("business", "similar",     "business")].edge_index
bb_ei2 = data[("business", "similar_rev", "business")].edge_index

adj_ub = make_norm_adj(ub_ei, NUM_USERS, NUM_BIZ ).to(DEVICE)
adj_bu = make_norm_adj(bu_ei, NUM_BIZ,   NUM_USERS).to(DEVICE)
adj_uu = make_norm_adj(uu_ei, NUM_USERS, NUM_USERS).to(DEVICE)

# Merge similar + similar_rev, then re-normalize
_idx  = torch.cat([
    torch.stack([bb_ei1[1].cpu(), bb_ei1[0].cpu()]),
    torch.stack([bb_ei2[1].cpu(), bb_ei2[0].cpu()]),
], dim=1)
_val  = torch.ones(_idx.size(1))
_adj_bb = torch.sparse_coo_tensor(_idx, _val, (NUM_BIZ, NUM_BIZ)).coalesce()
_row = _adj_bb.indices()[1]; _col = _adj_bb.indices()[0]
_deg_dst = torch.zeros(NUM_BIZ).scatter_add_(0, _col, torch.ones(_col.size(0)))
_deg_src = torch.zeros(NUM_BIZ).scatter_add_(0, _row, torch.ones(_row.size(0)))
_val_n   = _deg_dst[_col].pow(-0.5).clamp(max=1e5) * _deg_src[_row].pow(-0.5).clamp(max=1e5)
adj_bb   = torch.sparse_coo_tensor(
    torch.stack([_col, _row]), _val_n, (NUM_BIZ, NUM_BIZ)
).coalesce().to(DEVICE)

print(f"  adj_ub : {adj_ub.shape}   nnz={adj_ub._nnz():,}")
print(f"  adj_bu : {adj_bu.shape}   nnz={adj_bu._nnz():,}")
print(f"  adj_uu : {adj_uu.shape}   nnz={adj_uu._nnz():,}")
print(f"  adj_bb : {adj_bb.shape}   nnz={adj_bb._nnz():,}")
print(f"  Built in {time.time()-_t:.1f}s")

# ── Init model ────────────────────────────────────────────────────────────────
model = HeteroLightGCN(
    in_dim     = IN_DIM,
    embed_dim  = EMBED_DIM,
    num_layers = NUM_LAYERS,
    dropout    = DROPOUT,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nHeteroLightGCN initialized.  Trainable params: {total_params:,}")
print(model)


In [ ]:
# ── 4. Dataset classes ────────────────────────────────────────────────────────

class SimDataset(Dataset):
    """Positive pairs (anchor, pos) từ edges_business_business_similar.txt"""
    def __init__(self, pairs: list[tuple[int,int]]):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        anchor, pos = self.pairs[idx]
        return torch.tensor(anchor, dtype=torch.long), \
               torch.tensor(pos,    dtype=torch.long)


class UserDataset(Dataset):
    """
    Triplet (user, biz_pos, biz_neg) từ UB pairs.
    hard_neg = similar neighbor của biz_pos mà user chưa rate.

    Curriculum được điều khiển qua hard_prob (float 0.0 → 1.0):
      hard_prob = 0.0  → luôn random neg
      hard_prob = 1.0  → luôn hard neg
      0 < hard_prob < 1 → mỗi sample tự coin flip (per-sample, tránh oscillation)
    """
    def __init__(
        self,
        ub_pairs:      list[tuple[int,int]],
        user_pos:      dict[int, set],
        hard_neg_map:  dict[int, list],
        num_biz:       int,
        hard_prob:     float = 0.0,
    ):
        self.pairs        = ub_pairs
        self.user_pos     = user_pos
        self.hard_neg_map = hard_neg_map
        self.num_biz      = num_biz
        self.hard_prob    = hard_prob   # float trong [0, 1]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        u_idx, b_pos = self.pairs[idx]

        if random.random() < self.hard_prob:
            candidates = [
                b for b in self.hard_neg_map.get(b_pos, [])
                if b not in self.user_pos[u_idx]
            ]
            b_neg = random.choice(candidates) if candidates \
                    else random.randint(0, self.num_biz - 1)
        else:
            b_neg = random.randint(0, self.num_biz - 1)
            while b_neg in self.user_pos[u_idx]:
                b_neg = random.randint(0, self.num_biz - 1)

        return (
            torch.tensor(u_idx, dtype=torch.long),
            torch.tensor(b_pos, dtype=torch.long),
            torch.tensor(b_neg, dtype=torch.long),
        )


# Khởi tạo datasets + dataloaders
sim_dataset  = SimDataset(sim_pairs)
user_dataset = UserDataset(ub_pairs, user_pos, hard_neg_map, NUM_BIZ, hard_prob=0.0)

sim_loader  = DataLoader(sim_dataset,  batch_size=BATCH_SIZE, shuffle=True,  drop_last=True,  num_workers=0)
user_loader = DataLoader(user_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True,  num_workers=0)

print(f"SimDataset  : {len(sim_dataset):,} pairs  →  {len(sim_loader):,} batches/epoch")
print(f"UserDataset : {len(user_dataset):,} pairs  →  {len(user_loader):,} batches/epoch")


In [ ]:
# ── 5. Loss functions ─────────────────────────────────────────────────────────

def info_nce_loss(anchor: torch.Tensor, pos: torch.Tensor, tau: float = 0.07) -> torch.Tensor:
    """
    InfoNCE loss (in-batch negatives).
    anchor, pos: (B × D) — đã L2-normalize.
    Positive pair = (anchor[i], pos[i]).
    Negatives     = tất cả pos[j] với j ≠ i trong batch.
    """
    B = anchor.size(0)
    # logits[i, j] = sim(anchor[i], pos[j]) / τ   shape: (B × B)
    logits = (anchor @ pos.T) / tau
    labels = torch.arange(B, device=anchor.device)   # positive ở diagonal
    loss = F.cross_entropy(logits, labels)
    return loss


def bpr_loss(user_h: torch.Tensor,
             biz_h:  torch.Tensor,
             u_idx:  torch.Tensor,
             b_pos:  torch.Tensor,
             b_neg:  torch.Tensor) -> torch.Tensor:
    """
    BPR loss: score(u, b+) > score(u, b-)
    Dùng dot product vì đã L2-normalize.
    """
    u_vec   = user_h[u_idx]          # (B × D)
    b_p_vec = biz_h[b_pos]           # (B × D)
    b_n_vec = biz_h[b_neg]           # (B × D)

    pos_score = (u_vec * b_p_vec).sum(dim=-1)   # (B,)
    neg_score = (u_vec * b_n_vec).sum(dim=-1)   # (B,)

    loss = -F.logsigmoid(pos_score - neg_score).mean()
    return loss


print("Loss functions defined: info_nce_loss, bpr_loss")


NameError: name 'torch' is not defined

In [ ]:
# ── 6. Training loop (supports resume) ────────────────────────────────────────

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP, gamma=LR_DECAY)
scaler    = GradScaler(device="cuda", enabled=AMP)

SEP = "=" * 75
best_loss = float("inf")
history   = {"l_sim": [], "l_user": [], "total": []}
start_epoch = 1


def set_curriculum(epoch: int) -> str:
    """
    Tăng dần hard_prob từ 0.0 → 1.0 trong mixed window.
    Tránh oscillation do flip toàn bộ epoch giữa random và hard.
    """
    if epoch < CURRICULUM_MIXED_START:
        user_dataset.hard_prob = 0.0
        return "random"
    elif epoch < CURRICULUM_HARD_START:
        # tăng tuyến tính từ 0 → 1 trong mixed window
        p = (epoch - CURRICULUM_MIXED_START) / (CURRICULUM_HARD_START - CURRICULUM_MIXED_START)
        user_dataset.hard_prob = p
        return f"mix({p:.2f})"
    else:
        user_dataset.hard_prob = 1.0
        return "hard"


def rebuild_hard_neg_map(biz_h: torch.Tensor, top_k: int = REBUILD_NEG_TOP_K) -> dict:
    """
    Xây lại hard_neg_map từ embedding biz_h hiện tại thay vì text-similarity tĩnh.
    Dùng Faiss IndexFlatIP (cosine vì biz_h đã L2-normalize).
    """
    import faiss
    vecs = biz_h.detach().cpu().float().numpy()
    faiss.normalize_L2(vecs)
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(vecs)
    _, I = index.search(vecs, top_k + 1)   # +1 để loại self
    new_map = defaultdict(list)
    for i, neighbors in enumerate(I):
        new_map[i] = [int(n) for n in neighbors if n != i]
    return new_map


def _find_latest_epoch_ckpt(ckpt_dir: Path):
    files = sorted(ckpt_dir.glob("epoch_*.pt"))
    if not files:
        return None
    return files[-1]


def try_resume():
    global start_epoch, best_loss, history

    resume_path = None
    if RESUME_FROM is not None:
        resume_path = Path(RESUME_FROM)
    elif AUTO_RESUME:
        resume_path = _find_latest_epoch_ckpt(CKPT_DIR)

    if resume_path is None or not resume_path.exists():
        print("No resume checkpoint found. Start from epoch 1.")
        return

    print(f"Resuming from: {resume_path}")
    ckpt = torch.load(resume_path, map_location=DEVICE, weights_only=False)

    model.load_state_dict(ckpt["model"])

    if "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    if "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    if "scaler" in ckpt and ckpt["scaler"] is not None:
        scaler.load_state_dict(ckpt["scaler"])

    start_epoch = int(ckpt.get("epoch", 0)) + 1
    best_loss   = float(ckpt.get("best_loss", ckpt.get("l_user", best_loss)))

    old_history = ckpt.get("history", None)
    if isinstance(old_history, dict):
        history = old_history

    print(f"Resume done. start_epoch={start_epoch}, best_loss={best_loss:.4f}")


# ── Cache feat tensors on device (fp32 — sparse matmul requires fp32) ─────────
user_feat = data["user"].x.to(DEVICE).float()
biz_feat  = data["business"].x.to(DEVICE).float()

try_resume()

print(SEP)
print("START TRAINING  —  HeteroLightGCN")
print(SEP)

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t_ep = time.time()

    # ── LR warm restart khi bước vào hard phase ───────────────────────────────
    if epoch == CURRICULUM_HARD_START:
        for pg in optimizer.param_groups:
            pg["lr"] = LR_HARD_RESTART
        print(f"  → LR warm restart tại epoch {epoch}: lr={LR_HARD_RESTART:.2e}")

    curriculum = set_curriculum(epoch)

    optimizer.zero_grad(set_to_none=True)

    # ── Single forward pass — NO autocast (sparse CUDA matmul = fp32 only) ───
    user_h, biz_h, user_proj, biz_proj = model(
        user_feat, biz_feat, adj_ub, adj_bu, adj_uu, adj_bb
    )

    # ── Accumulate losses over N steps ───────────────────────────────────────
    n_steps = min(len(sim_loader), len(user_loader), MAX_STEPS_EPOCH)
    sim_iter  = iter(sim_loader)
    user_iter = iter(user_loader)

    run_l_sim  = 0.0
    run_l_user = 0.0
    accum_loss = torch.tensor(0.0, device=DEVICE)

    for _ in range(n_steps):
        anchor_idx, pos_idx          = next(sim_iter)
        u_idx, b_pos_idx, b_neg_idx  = next(user_iter)

        anchor_idx = anchor_idx.to(DEVICE, non_blocking=True)
        pos_idx    = pos_idx.to(DEVICE,    non_blocking=True)
        u_idx      = u_idx.to(DEVICE,      non_blocking=True)
        b_pos_idx  = b_pos_idx.to(DEVICE,  non_blocking=True)
        b_neg_idx  = b_neg_idx.to(DEVICE,  non_blocking=True)

        with autocast(device_type="cuda", enabled=AMP):
            l_sim  = info_nce_loss(biz_proj[anchor_idx], biz_proj[pos_idx], tau=TEMPERATURE)
            l_user = bpr_loss(user_h, biz_h, u_idx, b_pos_idx, b_neg_idx)
            step   = LAMBDA_SIM * l_sim + LAMBDA_USER * l_user

        accum_loss = accum_loss + step
        run_l_sim  += l_sim.item()
        run_l_user += l_user.item()

    avg_l_sim  = run_l_sim  / n_steps
    avg_l_user = run_l_user / n_steps

    # ── Backward ──────────────────────────────────────────────────────────────
    scaler.scale(accum_loss / n_steps).backward()
    scaler.unscale_(optimizer)
    grad_norm = clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()

    del user_h, biz_h, user_proj, biz_proj, accum_loss
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    elapsed    = time.time() - t_ep

    total_loss = LAMBDA_SIM * avg_l_sim + LAMBDA_USER * avg_l_user
    history.setdefault("l_sim", []).append(avg_l_sim)
    history.setdefault("l_user", []).append(avg_l_user)
    history.setdefault("total", []).append(total_loss)

    star = " ★BEST" if avg_l_user < best_loss else ""
    print(
        f"Epoch {epoch:>3}/{EPOCHS}"
        f" | L_sim={avg_l_sim:.4f}"
        f" | L_user={avg_l_user:.4f}"
        f" | total={total_loss:.4f}"
        f" | neg={curriculum:<12}"
        f" | lr={current_lr:.2e}"
        f" | grad={grad_norm:.3f}"
        f" | {elapsed:.1f}s{star}"
    )

    if epoch % 10 == 0 and torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        rsrvd = torch.cuda.memory_reserved()  / 1e9
        print(f"  [GPU] allocated={alloc:.2f}GB  reserved={rsrvd:.2f}GB")

    # ── Dynamic hard negative rebuild ─────────────────────────────────────────
    if epoch % REBUILD_NEG_EVERY == 0 and epoch >= CURRICULUM_MIXED_START:
        with torch.no_grad():
            _, biz_h_snap, _, _ = model(user_feat, biz_feat, adj_ub, adj_bu, adj_uu, adj_bb)
        hard_neg_map.clear()
        hard_neg_map.update(rebuild_hard_neg_map(biz_h_snap))
        user_dataset.hard_neg_map = hard_neg_map
        del biz_h_snap
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"  → Rebuilt hard_neg_map từ biz embedding hiện tại")

    if avg_l_user < best_loss:
        best_loss = avg_l_user
        torch.save(
            {
                "epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict() if AMP else None,
                "l_user": avg_l_user,
                "best_loss": best_loss,
                "history": history,
            },
            CKPT_DIR / "best.pt",
        )

    if epoch % SAVE_EVERY == 0:
        path = CKPT_DIR / f"epoch_{epoch:03d}.pt"
        torch.save(
            {
                "epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict() if AMP else None,
                "l_user": avg_l_user,
                "best_loss": best_loss,
                "history": history,
            },
            path,
        )
        print(f"  → Saved {path.name}")

print(SEP)
print(f"Training complete. Best L_user = {best_loss:.4f}")
print(SEP)


In [ ]:
# ── Loss curve ────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
xs = range(1, len(history["total"]) + 1)
ax.plot(xs, history["l_sim"],  label="L_sim  (InfoNCE)")
ax.plot(xs, history["l_user"], label="L_user (BPR)")
ax.plot(xs, history["total"],  label="Total",  linestyle="--")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("HeteroLightGCN Training Loss")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── 8. Quick evaluation metrics after training (LOAD CHECKPOINT) ──────────────
import torch
import torch.nn.functional as F
import random
import math
from collections import defaultdict
from pathlib import Path

CKPT_PATH = "/content/chkpt6/epoch_180.pt"

# ── 1. Load model từ checkpoint ───────────────────────────────────────────────
def load_model_from_ckpt(model, ckpt_path, device):
    print(f"Loading checkpoint from: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device)

    model.load_state_dict(ckpt["model"])
    model.eval()

    print(f"Loaded epoch: {ckpt.get('epoch', 'N/A')}")
    print(f"Best loss: {ckpt.get('best_loss', 'N/A')}")
    return model


# ── 2. Get embeddings ─────────────────────────────────────────────────────────
@torch.no_grad()
def get_final_embeddings(model):
    model.eval()
    u_h, b_h, _, _ = model(user_feat, biz_feat, adj_ub, adj_bu, adj_uu, adj_bb)
    return F.normalize(u_h, dim=-1), F.normalize(b_h, dim=-1)


# ── 3. User ranking evaluation ───────────────────────────────────────────────
def evaluate_user_ranking(
    user_h: torch.Tensor,
    biz_h: torch.Tensor,
    user_pos_map: dict,
    num_biz: int,
    k_list=(5, 10, 20),
    num_eval_users=5000,
    num_neg=100,
    min_pos_per_user=3,
    seed=42,
):
    rng = random.Random(seed)

    eligible_users = [u for u, items in user_pos_map.items() if len(items) >= min_pos_per_user]
    if not eligible_users:
        raise ValueError("No eligible users for evaluation.")

    n_eval = min(num_eval_users, len(eligible_users))
    eval_users = rng.sample(eligible_users, n_eval)

    hit = {k: 0.0 for k in k_list}
    ndcg = {k: 0.0 for k in k_list}
    mrr = 0.0
    auc = 0.0

    for u in eval_users:
        pos_items = list(user_pos_map[u])
        gt_item = rng.choice(pos_items)
        seen = set(pos_items)
        seen.discard(gt_item)

        neg_items = []
        while len(neg_items) < num_neg:
            n = rng.randrange(num_biz)
            if n != gt_item and n not in seen:
                neg_items.append(n)

        cand = [gt_item] + neg_items

        scores = torch.mv(biz_h[cand], user_h[u]).cpu()

        order = torch.argsort(scores, descending=True)
        rank = (order == 0).nonzero(as_tuple=True)[0].item() + 1

        mrr += 1.0 / rank
        auc += (scores[0] > scores[1:]).float().mean().item()

        for k in k_list:
            if rank <= k:
                hit[k] += 1.0
                ndcg[k] += 1.0 / math.log2(rank + 1)

    out = {
        "n_eval_users": n_eval,
        "AUC": auc / n_eval,
        "MRR": mrr / n_eval,
    }
    for k in k_list:
        out[f"HR@{k}"] = hit[k] / n_eval
        out[f"NDCG@{k}"] = ndcg[k] / n_eval
    return out


# ── 4. Business similarity evaluation ─────────────────────────────────────────
def evaluate_business_similarity(
    biz_h: torch.Tensor,
    sim_pairs_list: list,
    k_list=(10, 20),
    num_eval_biz=3000,
    seed=42,
):
    rng = random.Random(seed)

    gt = defaultdict(set)
    for a, p in sim_pairs_list:
        gt[a].add(p)

    eligible = [b for b, neigh in gt.items() if len(neigh) > 0]
    if not eligible:
        raise ValueError("No eligible businesses.")

    n_eval = min(num_eval_biz, len(eligible))
    eval_biz = rng.sample(eligible, n_eval)

    sim_mat = biz_h @ biz_h.T   # ⚠️ nếu biz lớn → có thể OOM

    recall = {k: 0.0 for k in k_list}
    for b in eval_biz:
        scores = sim_mat[b].clone()
        scores[b] = -1e9

        max_k = max(k_list)
        topk = torch.topk(scores, k=max_k).indices.tolist()

        pos_set = gt[b]
        for k in k_list:
            hit_k = len(set(topk[:k]) & pos_set)
            recall[k] += hit_k / max(len(pos_set), 1)

    out = {"n_eval_businesses": n_eval}
    for k in k_list:
        out[f"Recall@{k}"] = recall[k] / n_eval
    return out


# ── 5. RUN ────────────────────────────────────────────────────────────────────

# load model weight
model = load_model_from_ckpt(model, CKPT_PATH, DEVICE)

print("\nComputing embeddings...")
user_h_eval, biz_h_eval = get_final_embeddings(model)

print("Running evaluation...")

user_metrics = evaluate_user_ranking(
    user_h=user_h_eval,
    biz_h=biz_h_eval,
    user_pos_map=user_pos,
    num_biz=NUM_BIZ,
    k_list=(5, 10, 20),
    num_eval_users=5000,
    num_neg=100,
    min_pos_per_user=3,
)

biz_metrics = evaluate_business_similarity(
    biz_h=biz_h_eval,
    sim_pairs_list=sim_pairs,
    k_list=(10, 20),
    num_eval_biz=3000,
)

print("\n[User-Business Ranking]")
for k, v in user_metrics.items():
    print(f"{k:12s}: {v:.4f}" if isinstance(v, float) else f"{k:12s}: {v}")

print("\n[Business Similarity Retrieval]")
for k, v in biz_metrics.items():
    print(f"{k:12s}: {v:.4f}" if isinstance(v, float) else f"{k:12s}: {v}")